In [64]:
from SPARQLWrapper import SPARQLWrapper, JSON
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
from rdflib import Graph, URIRef, RDFS
import csv
import re
from collections import Counter, defaultdict
import csv
import requests
import time
import os
import urllib.parse
from bs4 import BeautifulSoup
from urllib.parse import unquote


In [71]:
#pip freeze > requirements.txt

In [3]:
# Set up the endpoint
endpoint_url = "https://query.wikidata.org/sparql"
sparql = SPARQLWrapper(endpoint_url)
sparql.setReturnFormat(JSON)

### Direct connections to instances of Holy Well/ Holy Well Semantic Concept

In [28]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

query = """
SELECT ?subject ?subjectLabel ?predicate ?propEntity ?propLabel ?object WHERE {
  ?subject wdt:P31 wd:Q126443332 .
  ?subject ?predicate ?object .
  
  # Map both wdt: and prop: URIs to the property entity
  BIND(
    IRI(
      REPLACE(
        STR(?predicate),
        "http://www.wikidata.org/prop(/direct)?/",
        "http://www.wikidata.org/entity/"
      )
    ) AS ?propEntity
  )

  SERVICE wikibase:label { 
    bd:serviceParam wikibase:language "en" .
    ?subject rdfs:label ?subjectLabel .
    ?propEntity rdfs:label ?propLabel .
  }
}
"""
sparql.setQuery(query)
results = sparql.query().convert()

# Convert to dataframe
triples = []
for result in results["results"]["bindings"]:
    triples.append({
        "subject_q": result["subject"]["value"].split("/")[-1],
        "subject_label": result.get("subjectLabel", {}).get("value", ""),
        "predicate_p": result["propEntity"]["value"].split("/")[-1],  # P-code entity
        "predicate_label": result.get("propLabel", {}).get("value", ""),  # English label
        "object": result["object"]["value"]
    })

df_triples = pd.DataFrame(triples)

# Keep only properties starting with P
df_triples = df_triples[df_triples["predicate_p"].str.startswith("P")]

# Save to CSV
df_triples.to_csv("holy_wells_semantic_concept.csv", index=False)

# Print unique subject count
print("Unique subject Q-codes:", df_triples["subject_q"].nunique())
print("Unique predicate P-codes:", df_triples["predicate_p"].nunique())
print("Unique objects:", df_triples["object"].nunique())


Unique subject Q-codes: 229
Unique predicate P-codes: 37
Unique objects: 7444


In [ ]:
import csv
import re
from collections import Counter
from SPARQLWrapper import SPARQLWrapper, JSON

# Input CSV
csv_file = 'holy_wells_semantic_concept.csv'

pcode_counter = Counter()
pattern = re.compile(r'(P\d+)')  # match P-codes anywhere in the column

with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        match = pattern.search(row['predicate_p'])
        if match:
            pcode = match.group(1)
            pcode_counter[pcode] += 1


def get_labels(pcodes):
    """Fetch English labels for a list of Wikidata properties."""
    labels = {}
    if not pcodes:
        return labels

    # Use wd:P... for the property itself
    values = " ".join(f"wd:{p}" for p in pcodes)
    query = f"""
    SELECT ?p ?pLabel WHERE {{
      VALUES ?p {{ {values} }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    for r in results["results"]["bindings"]:
        prop_uri = r["p"]["value"]
        pcode = prop_uri.split("/")[-1]
        label = r["pLabel"]["value"]
        labels[pcode] = label
    return labels

# Step 3: Fetch labels for all P-codes found
all_pcodes = list(pcode_counter.keys())
pcode_to_label = get_labels(all_pcodes)

# Step 4: Print most used predicates
print(f"{'P-code':<8} {'Count':<6} Label")
print("-" * 40)
for pcode, count in pcode_counter.most_common():
    label = pcode_to_label.get(pcode, "[Label not found]")
    print(f"{pcode:<8} {count:<6} {label}")


P-code   Count  Label
----------------------------------------
P1343    3788   described by source
P31      1177   instance of
P131     1118   located in the administrative territorial entity
P217     758    inventory number
P195     748    collection
P17      458    country
P625     432    coordinate location
P138     402    named after
P708     384    diocese
P4057    339    Irish Sites and Monuments Record ID
P11693   280    OpenStreetMap node ID
P18      173    image
P6375    155    street address
P373     140    Commons category
P973     92     described at URL
P841     72     feast day
P1325    66     external data available at URL
P2186    66     Wiki Loves Monuments ID
P10      40     video
P1651    38     YouTube video ID
P403     16     mouth of the watercourse
P2175    12     medical condition treated
P10689   12     OpenStreetMap way ID
P110     10     illustrator
P7959    10     historic county
P4088    6      Irish National Inventory of Architectural Heritage ID
P1050    

In [12]:
#todo: P51      1      audio - works like video or image but also really?? 
#todo: P2441    1      literal translation
#todo: P571     1      inception
#todo: P1801    1      plaque image -- difference to image??

In [ ]:
# query to get en label of property
# SELECT ?property ?label WHERE {
# BIND(wd:P31 AS ?property)
# ?property rdfs:label ?label .
# FILTER(LANG(?label) = "en")}

In [29]:
#Get entity with most unique direct predicates -columbkile's well
entity_query = """
SELECT ?item (COUNT(DISTINCT ?prop) AS ?propCount) WHERE {
  ?item wdt:P31 wd:Q126443332 .
  ?item ?prop ?val .
}
GROUP BY ?item
ORDER BY DESC(?propCount)
LIMIT 1
"""

sparql.setQuery(entity_query)
results_most_unique_predicates = sparql.query().convert()
top_entity = results_most_unique_predicates["results"]["bindings"][0]["item"]["value"]

print(f"Entity with most unique predicates: {top_entity}")


Entity with most unique predicates: http://www.wikidata.org/entity/Q126456441


#### Columbkille's Well

In [15]:
target_qcode = "Q126456441" # columbkille's well
csv_file = 'holy_wells_semantic_concept.csv'
qcode_labels_file = 'qcode_labels_holy_wells_semantic_concept.csv'
pcode_labels_file = 'pcode_labels_holy_wells_semantic_concept.csv'

## Functions to fetch a star around a subject

### Instance of

In [35]:
def get_instance_of(qid: str) -> pd.DataFrame:
    """
    Fetch all 'instance of' (P31) statements for a given Wikidata entity,
    returning subject Q, instance-of Q + label, and stated-in Q + label (if present).
    """

    query = f"""
    SELECT ?instanceOf ?instanceOfLabel ?statedIn ?statedInLabel
    WHERE {{
      wd:{qid} p:P31 ?stmt .
      ?stmt ps:P31 ?instanceOf .

      OPTIONAL {{
        ?instanceOf rdfs:label ?instanceOfLabel .
        FILTER(LANG(?instanceOfLabel) = "en")
      }}

      OPTIONAL {{
        ?stmt prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?statedIn .
        OPTIONAL {{
          ?statedIn rdfs:label ?statedInLabel .
          FILTER(LANG(?statedInLabel) = "en")
        }}
      }}
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for b in results["results"]["bindings"]:
        rows.append({
            "subject_q": qid,
            "instanceOf_q": b.get("instanceOf", {}).get("value", "").split("/")[-1],
            "instanceOf": b.get("instanceOfLabel", {}).get("value", ""),
            "stated_in_q": b.get("statedIn", {}).get("value", "").split("/")[-1] if "statedIn" in b else "",
            "stated_in": b.get("statedInLabel", {}).get("value", ""),
        })

    return pd.DataFrame(rows)

In [37]:
df = get_instance_of("Q126456441")
df.to_csv("instance_of.csv", index=False)

### Description, aliases, language

In [ ]:
#get description, aliases, language
def fetch_description_alias_language(subject_qcode: str) -> pd.DataFrame:
    query = f"""
    SELECT ?lang ?valueType ?value WHERE {{
      VALUES ?item {{ wd:{subject_qcode} }}
      {{
        ?item rdfs:label ?value .
        BIND("label" AS ?valueType)
        BIND(LANG(?value) AS ?lang)
      }}
      UNION
      {{
        ?item schema:description ?value .
        BIND("description" AS ?valueType)
        BIND(LANG(?value) AS ?lang)
      }}
      UNION
      {{
        ?item skos:altLabel ?value .
        BIND("alias" AS ?valueType)
        BIND(LANG(?value) AS ?lang)
      }}
    }}
    ORDER BY ?lang ?valueType
    """
    sparql.setQuery(query)
    results = sparql.query().convert()
    
    # Collect descriptions and aliases per language
    data = defaultdict(lambda: {"description": "", "aliases": []})
    for binding in results["results"]["bindings"]:
        lang = binding["lang"]["value"]
        vtype = binding["valueType"]["value"]
        val   = binding["value"]["value"]
        if vtype == "description":
            data[lang]["description"] = val
        elif vtype == "alias":
            data[lang]["aliases"].append(val)
    
    # Build final rows
    rows = []
    for lang, content in data.items():
        desc = content["description"]
        if content["aliases"]:
            for alias in content["aliases"]:
                rows.append({
                    "subject": subject_qcode,
                    "language": lang,
                    "description": desc,
                    "alias": alias
                })
        else:
            rows.append({
                "subject_q": subject_qcode,
                "language": lang,
                "description": desc,
                "alias": ""
            })
    
    return pd.DataFrame(rows, columns=["subject", "language", "description", "alias"])



In [40]:
df = fetch_description_alias_language(target_qcode)
df.to_csv("lang_description_aliases.csv", index=False)


### statements and references - Toberbride, St Bridgets, Kenny's Well as example

In [41]:
#illustrator, work and their instance of - modelled in protege. could get the meta for the monograph too.
def get_illustrator_references(subject_q):
    query = f"""
    SELECT ?illustrator ?illustratorLabel 
           ?illustratorInstance ?illustratorInstanceLabel
           ?statedIn ?statedInLabel 
           ?statedInInstanceOf ?statedInInstanceOfLabel 
    WHERE {{
      wd:{subject_q} p:P110 ?statement .
      ?statement ps:P110 ?illustrator .

      OPTIONAL {{
        ?illustrator wdt:P31 ?illustratorInstance .
      }}

      OPTIONAL {{
        ?statement prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?statedIn .
        
        OPTIONAL {{
          ?statedIn p:P31 ?instanceStatement .
          ?instanceStatement ps:P31 ?statedInInstanceOf .
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    records = []
    for b in results["results"]["bindings"]:
        records.append({
            "subject_q": subject_q,
            "illustrator_q": b["illustrator"]["value"].split("/")[-1],
            "illustrator_label": b.get("illustratorLabel", {}).get("value", ""),
            "illustrator_instance_q": b.get("illustratorInstance", {}).get("value", "").split("/")[-1] if "illustratorInstance" in b else "",
            "illustrator_instance_label": b.get("illustratorInstanceLabel", {}).get("value", "") if "illustratorInstanceLabel" in b else "",
            "stated_in_q": b.get("statedIn", {}).get("value", "").split("/")[-1] if "statedIn" in b else "",
            "stated_in_label": b.get("statedInLabel", {}).get("value", "") if "statedInLabel" in b else "",
            "stated_in_instance_q": b.get("statedInInstanceOf", {}).get("value", "").split("/")[-1] if "statedInInstanceOf" in b else "",
            "stated_in_instance_of_label": b.get("statedInInstanceOfLabel", {}).get("value", "") if "statedInInstanceOfLabel" in b else "",
        })

    return pd.DataFrame(records)


In [42]:
#Toberbride as example
df = get_illustrator_references("Q122222699")
df.to_csv("illustrator.csv", index=False)

In [43]:
# state and metadata - modeled in protege
def fetch_state_qualifiers_with_source_metadata(well_qid):
    from SPARQLWrapper import SPARQLWrapper, JSON
    import pandas as pd

    # Main query
    query = f"""
    SELECT ?qualifier_property ?qualifier_value ?label ?description 
           ?stated_in_qid ?stated_in_label ?source_prop ?source_value ?source_value_label WHERE {{
      BIND(wd:{well_qid} AS ?well)

      ?well p:P31 ?stmt .
      ?stmt ?pq ?qualifier_value .
      FILTER(?pq IN (pq:P5817, pq:P5816))  # state of use, state of conservation

      BIND(STR(REPLACE(STR(?pq), "^.*(P\\\\d+)$", "$1")) AS ?qualifier_property)

      OPTIONAL {{
        ?qualifier_value rdfs:label ?label .
        FILTER(LANG(?label) = "en")
      }}

      OPTIONAL {{
        ?qualifier_value schema:description ?description .
        FILTER(LANG(?description) = "en")
      }}

      OPTIONAL {{
        ?stmt prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?stated_in .
        BIND(STRAFTER(STR(?stated_in), "entity/") AS ?stated_in_qid)

        OPTIONAL {{
          ?stated_in rdfs:label ?stated_in_label .
          FILTER(LANG(?stated_in_label) = "en")
        }}

        OPTIONAL {{
          ?stated_in ?source_pred_uri ?source_value .
          FILTER(STRSTARTS(STR(?source_pred_uri), "http://www.wikidata.org/prop/direct/"))
          BIND(STRAFTER(STR(?source_pred_uri), "/prop/direct/") AS ?source_prop)

          OPTIONAL {{
            ?source_value rdfs:label ?source_value_label .
            FILTER(LANG(?source_value_label) = "en")
          }}
        }}
      }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = {}
    all_source_props = set()
    all_source_qid_props = set()

    for b in results["results"]["bindings"]:
        key = (
            well_qid,
            b.get("qualifier_property", {}).get("value", ""),
            b.get("qualifier_value", {}).get("value", ""),
            b.get("stated_in_qid", {}).get("value", "")
        )

        if key not in rows:
            rows[key] = {
                "subject_q": well_qid,
                "predicate_p": b.get("qualifier_property", {}).get("value", ""),
                "object_q": b.get("qualifier_value", {}).get("value", "").split("/")[-1],
                "object": b.get("label", {}).get("value", "") or b.get("description", {}).get("value", ""),
                "stated_in_q": b.get("stated_in_qid", {}).get("value", ""),
                "stated_in_label": b.get("stated_in_label", {}).get("value", "")
            }

        source_prop = b.get("source_prop", {}).get("value")
        source_value_uri = b.get("source_value", {}).get("value")
        source_value_label = b.get("source_value_label", {}).get("value")

        if source_prop:
            # Add label
            prop_value = source_value_label or source_value_uri
            if prop_value:
                rows[key][source_prop] = prop_value
                all_source_props.add(source_prop)

            # Add QID
            if source_value_uri and "entity/" in source_value_uri:
                qid = source_value_uri.split("/")[-1]
                qid_key = f"{source_prop}_q"
                rows[key][qid_key] = qid
                all_source_qid_props.add(qid_key)

    # Fetch property labels
    all_p_codes = all_source_props.union(p.replace("_q", "") for p in all_source_qid_props)
    prop_labels = {}

    for p_code in all_p_codes:
        sparql.setQuery(f"""
        SELECT ?label WHERE {{
          wd:{p_code} rdfs:label ?label .
          FILTER(LANG(?label) = "en")
        }} LIMIT 1
        """)
        try:
            res = sparql.query().convert()
            label = res["results"]["bindings"][0]["label"]["value"]
            prop_labels[p_code] = label
        except Exception:
            prop_labels[p_code] = p_code  # fallback to P-code if label not found

    # Build final column headers
    standard_headers = ["subject_q", "predicate_p", "object_q", "object", "stated_in_q", "stated_in_label"]

    grouped_source_headers = []
    for p_code in sorted(all_p_codes):
        label = prop_labels[p_code]
        grouped_source_headers.extend([label, f"{label}_q"])

    all_headers = standard_headers + grouped_source_headers

    # Normalize rows
    normalized_rows = []
    for row in rows.values():
        norm_row = {h: "" for h in all_headers}
        for h in standard_headers:
            norm_row[h] = row.get(h, "")

        for p_code in all_p_codes:
            label = prop_labels[p_code]
            label_col = label
            qid_col = f"{label}_q"

            norm_row[label_col] = row.get(p_code, "")
            norm_row[qid_col] = row.get(f"{p_code}_q", "")

        normalized_rows.append(norm_row)

    return pd.DataFrame(normalized_rows)


In [44]:
# Toberbride as example
df = fetch_state_qualifiers_with_source_metadata("Q122258940")
df.to_csv("state.csv", index=False)

In [45]:
#street address 
def get_street_address_references(subject_q):
    query = f"""
    SELECT DISTINCT ?streetText ?statedIn ?statedInLabel ?statedInInstanceOf ?statedInInstanceOfLabel WHERE {{
      wd:{subject_q} p:P6375 ?statement .
      ?statement ps:P6375 ?streetText .

      OPTIONAL {{
        ?statement prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?statedIn .

        OPTIONAL {{
          ?statedIn p:P31 ?instanceStatement .
          ?instanceStatement ps:P31 ?statedInInstanceOf .
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    records = []
    for b in results["results"]["bindings"]:
        records.append({
            "subject_q": subject_q,
            "street_address": b.get("streetText", {}).get("value", ""),
            "stated_in_q": b.get("statedIn", {}).get("value", "").split("/")[-1] if "statedIn" in b else "",
            "stated_in_label": b.get("statedInLabel", {}).get("value", "") if "statedInLabel" in b else "",
            "stated_in_instance_q": b.get("statedInInstanceOf", {}).get("value", "").split("/")[-1] if "statedInInstanceOf" in b else "",
            "stated_in_instance_label": b.get("statedInInstanceOfLabel", {}).get("value", "") if "statedInInstanceOfLabel" in b else "",
        })

    return pd.DataFrame(records)


In [46]:
# St. Bridget's Well as example
df = get_street_address_references("Q126454471")
df.to_csv("street_addr.csv", index=False)

#### URL - Kenny's Well

In [47]:
# modelled in protege - add a column with a copy of the url with all / replaced with _ 
def get_url_meta(subject_q, predicate_p):
    query = f"""
    PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?url ?qualifierProp ?qualifierPropLabel ?qualifierValue ?qualifierValueLabel ?qualifierValueP31 ?qualifierValueP31Label WHERE {{
      wd:{subject_q} p:{predicate_p} ?statement .
      ?statement ps:{predicate_p} ?url .

      OPTIONAL {{
        ?statement ?qualifierProp ?qualifierValue .
        FILTER(STRSTARTS(STR(?qualifierProp), STR(pq:)))

        # Get label of qualifier property
        BIND(IRI(CONCAT("http://www.wikidata.org/entity/",
            STRAFTER(STR(?qualifierProp), "http://www.wikidata.org/prop/qualifier/"))) AS ?propEntity)
        ?propEntity rdfs:label ?qualifierPropLabel .
        FILTER(LANG(?qualifierPropLabel) = "en")

        # If qualifierValue is an entity, get its P31 (instance of) and label
        OPTIONAL {{
          FILTER(STRSTARTS(STR(?qualifierValue), STR(wd:)))
          ?qualifierValue wdt:P31 ?qualifierValueP31 .
          ?qualifierValueP31 rdfs:label ?qualifierValueP31Label .
          FILTER(LANG(?qualifierValueP31Label) = "en")
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for res in results["results"]["bindings"]:
        url = res["url"]["value"]
        qualifierPropLabel = res.get("qualifierPropLabel", {}).get("value")
        qualifierValueLabel = res.get("qualifierValueLabel", {}).get("value")
        qualifierValueRaw = res.get("qualifierValue", {}).get("value")
        
        # Prefer the label of the qualifier value, fallback to raw value (URI or literal)
        qualifierValue = qualifierValueLabel if qualifierValueLabel else qualifierValueRaw
        
        qualifierValueP31 = res.get("qualifierValueP31", {}).get("value")
        # Extract Q code from the full URI, if present
        if qualifierValueP31:
            qualifierValueP31 = qualifierValueP31.split("/")[-1]
        qualifierValueP31Label = res.get("qualifierValueP31Label", {}).get("value")

        rows.append({
            "subject_q":subject_q,
            "url": url,
            "qualifier_property": qualifierPropLabel,
            "qualifier_value": qualifierValue,
            "qualifier_value_p31": qualifierValueP31,
            "qualifier_value_p31_label": qualifierValueP31Label
        })
    df = pd.DataFrame(rows)
    return df


In [48]:
def get_url_meta(subject_q, predicate_p):
    query = f"""
    PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?url ?qualifierProp ?qualifierPropLabel ?qualifierValue ?qualifierValueLabel ?qualifierValueP31 ?qualifierValueP31Label WHERE {{
      wd:{subject_q} p:{predicate_p} ?statement .
      ?statement ps:{predicate_p} ?url .

      OPTIONAL {{
        ?statement ?qualifierProp ?qualifierValue .
        FILTER(STRSTARTS(STR(?qualifierProp), STR(pq:)))

        # Get label of qualifier property
        BIND(IRI(CONCAT("http://www.wikidata.org/entity/",
            STRAFTER(STR(?qualifierProp), "http://www.wikidata.org/prop/qualifier/"))) AS ?propEntity)
        ?propEntity rdfs:label ?qualifierPropLabel .
        FILTER(LANG(?qualifierPropLabel) = "en")

        # If qualifierValue is an entity, get its P31 (instance of) and label
        OPTIONAL {{
          FILTER(STRSTARTS(STR(?qualifierValue), STR(wd:)))
          ?qualifierValue wdt:P31 ?qualifierValueP31 .
          ?qualifierValueP31 rdfs:label ?qualifierValueP31Label .
          FILTER(LANG(?qualifierValueP31Label) = "en")
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for res in results["results"]["bindings"]:
        url = res["url"]["value"]
        qualifierPropLabel = res.get("qualifierPropLabel", {}).get("value")
        qualifierValueLabel = res.get("qualifierValueLabel", {}).get("value")
        qualifierValueRaw = res.get("qualifierValue", {}).get("value")
        
        # Prefer the label of the qualifier value, fallback to raw value (URI or literal)
        qualifierValue = qualifierValueLabel if qualifierValueLabel else qualifierValueRaw
        
        qualifierValueP31 = res.get("qualifierValueP31", {}).get("value")
        # Extract Q code from the full URI, if present
        if qualifierValueP31:
            qualifierValueP31 = qualifierValueP31.split("/")[-1]
        qualifierValueP31Label = res.get("qualifierValueP31Label", {}).get("value")

        # Add new column with URL where all '/' replaced with '_'
        url_underscore = url.replace("/", "_")

        rows.append({
            "subject_q": subject_q,
            "url": url,
            "url_underscore": url_underscore,
            "qualifier_property": qualifierPropLabel,
            "qualifier_value": qualifierValue,
            "qualifier_value_p31": qualifierValueP31,
            "qualifier_value_p31_label": qualifierValueP31Label
        })
    df = pd.DataFrame(rows)
    return df


In [ ]:
#described at URL - Kenny's Well
df = get_url_meta("Q114439798", "P973")
df.to_csv("url_meta.csv", index=False)

In [ ]:
#external data available - Kenny's Well
df = get_url_meta("Q114439798", "P1325")
df.to_csv("url_external.csv", index=False)

In [50]:
# non-free artwork image URL
df = get_url_meta("Q126472997", "P6500")
df.to_csv("url_nonfree.csv", index=False)

#### Inventory-Collection - Columbkille's Well

In [51]:
#inventory number with collection - modeled in protege
def fetch_well_inventory(target_qcode: str) -> pd.DataFrame:
    query = f"""
    SELECT ?well ?invNum ?collection ?label WHERE {{
      BIND(wd:{target_qcode} AS ?well)

      ?well wdt:P31 wd:Q126443332 ;       # Ensure it's a holy well semantic concept
            p:P217 ?stmt .                # Get inventory number statement node

      ?stmt ps:P217 ?invNum .             # The inventory number value
      ?stmt pq:P195 ?collection .         # The associated collection (qualifier)

      OPTIONAL {{
        ?collection rdfs:label ?label .   # Collection label
        FILTER (lang(?label) = "en")
      }}
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()
    
    # Parse bindings into rows
    rows = [
        {
            "subject_q": target_qcode,
            "inventory_id": binding["invNum"]["value"],
            "collection_q": binding["collection"]["value"].rsplit("/", 1)[-1],
            "collection_label": binding.get("label", {}).get("value", "")
        }
        for binding in results["results"]["bindings"]
    ]

    # Create DataFrame
    df = pd.DataFrame(rows)
    return df


In [ ]:
df = fetch_well_inventory(target_qcode)
df.to_csv("inventory_id_coll.csv", index=False)

#### YT Video and metadata - Columbkille as example

In [54]:
#YouTube video ID, subject named as, nr of views, publication date, duration, yt channel id, point in time
#modeled in protege
def fetch_well_yt_videos(target_qcode: str) -> pd.DataFrame:
    query = f"""
    SELECT ?vidID ?date ?dur WHERE {{
      wd:{target_qcode} p:P1651 ?stmt .
      ?stmt ps:P1651 ?vidID .
      OPTIONAL {{ ?stmt pq:P577  ?date }}
      OPTIONAL {{ ?stmt pq:P2047 ?dur  }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for b in results["results"]["bindings"]:
        rows.append({
            "subject_q":          target_qcode,
            "youtube_id":        b["vidID"]["value"],
            "publication_date":  b.get("date", {}).get("value"),
            "duration":          b.get("dur",  {}).get("value"),
        })

    return pd.DataFrame(rows, columns=[
        "subject_q", "youtube_id", "publication_date", "duration"
    ])


In [56]:
df = fetch_well_yt_videos(target_qcode)
df.to_csv("YouTube_Video_ID_Metadata.csv", index=False)

#### Image and Video metadata from wiki commons - Columbkille as example

In [67]:
def extract_name_and_link(html_string):
    if not html_string:
        return "", ""
    soup = BeautifulSoup(html_string, "html.parser")
    tag = soup.find("a")
    if tag:
        return tag.get_text(strip=True), f"https:{tag['href']}" if tag['href'].startswith("//") else tag['href']
    else:
        return soup.get_text(strip=True), ""


In [83]:
import requests
import pandas as pd
from urllib.parse import unquote
from bs4 import BeautifulSoup


def get_meta(subject_q, predicate_p):
    """Fetch image metadata for a given Wikidata subject (Q-code) and predicate (e.g. P18).

    - Normalizes filenames for Wikimedia Commons by decoding percent-encodings and
      replacing spaces with underscores (DO NOT percent-encode apostrophes etc.).
    - Adds a User-Agent header to avoid some automated-blocking rules.
    - Handles non-200 / non-JSON responses and logs helpful messages instead of
      raising JSONDecodeError.
    """
    # Build and run SPARQL query (assumes `sparql` object is available in scope)
    query = f"""
    SELECT ?image WHERE {{
      wd:{subject_q} wdt:{predicate_p} ?image .
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()
    bindings = results.get("results", {}).get("bindings", [])

    if not bindings:
        return pd.DataFrame(columns=["Q", "file_url", "commons_file_url", "wikimedia_url"])

    rows = []

    # Set a polite User-Agent. Replace with your project/contact URL if available.
    headers = {"User-Agent": "MetadataFetcher/1.0 (https://example.com)"}

    for binding in bindings:
        image_url = binding["image"]["value"]
        filename = image_url.split("/")[-1]

        # Normalize filename for Commons: decode percent-escapes, convert spaces to underscores.
        decoded_filename = unquote(filename).replace(" ", "_")
        commons_filename = f"File:{decoded_filename}"

        api_url = "https://commons.wikimedia.org/w/api.php"
        params = {
            "action": "query",
            "prop": "imageinfo",
            "titles": commons_filename,
            "iiprop": "url|extmetadata",
            "format": "json",
        }

        try:
            response = requests.get(api_url, params=params, headers=headers, timeout=15)
        except requests.RequestException as e:
            print(f"Request failed for {commons_filename}: {e}")
            continue

        # Helpful logging for common HTTP statuses
        if response.status_code == 403:
            print(f"Wikimedia API error 403 for {commons_filename} — check filename normalization or access restrictions")
            continue
        if response.status_code != 200:
            print(f"Wikimedia API error {response.status_code} for {commons_filename}")
            continue

        content_type = response.headers.get("Content-Type", "")
        if "application/json" not in content_type:
            # Sometimes the API returns an HTML error page or empty body.
            preview = response.text[:400].replace("", " ")
            print(f"Non-JSON response for {commons_filename}. Preview: {preview}")
            continue

        try:
            data = response.json()
        except ValueError as e:
            print(f"JSON decode failed for {commons_filename}: {e}")
            print(f"Response preview: {response.text[:400]}")
            continue

        pages = data.get("query", {}).get("pages", {})
        for page_id, page_data in pages.items():
            imageinfo = page_data.get("imageinfo", [{}])[0]
            wikimedia_url = imageinfo.get("url", "")
            extmetadata = imageinfo.get("extmetadata", {})

            row = {
                "Q": subject_q,
                "file_url": image_url,
                "commons_file_url": f"https://commons.wikimedia.org/wiki/{commons_filename}",
                "wikimedia_url": wikimedia_url,
            }

            # Copy extmetadata fields (except Categories)
            for key, meta in extmetadata.items():
                if key != "Categories":
                    row[key] = meta.get("value", "")

            # Extract Artist and Credit into name/link pairs
            artist_html = extmetadata.get("Artist", {}).get("value", "")
            artist_name, artist_link = extract_name_and_link(artist_html)
            row["artist_name"] = artist_name
            row["artist_link"] = artist_link

            credit_html = extmetadata.get("Credit", {}).get("value", "")
            credit_name, credit_link = extract_name_and_link(credit_html)
            row["credit_name"] = credit_name
            row["credit_link"] = credit_link

            # Remove raw fields we already processed
            row.pop("Artist", None)
            row.pop("Credit", None)
            row.pop("Categories", None)

            rows.append(row)

    return pd.DataFrame(rows)


In [84]:
#image and metadata
df = get_meta(target_qcode, "P18")
df.to_csv("image_meta.csv", index=False)

In [85]:
#video and metadata
df = get_meta(target_qcode, "P10")
df.to_csv("video_meta.csv", index=False)

### direct connections

#### Coordinates and point - Columbkille example

In [77]:
def dms_string(lat, lon):
    def to_dms(deg):
        d = int(deg)
        m_float = abs((deg - d) * 60)
        m = int(m_float)
        s = round((m_float - m) * 60, 4)
        return d, m, s

    lat_d, lat_m, lat_s = to_dms(abs(lat))
    lon_d, lon_m, lon_s = to_dms(abs(lon))

    lat_hem = "N" if lat >= 0 else "S"
    lon_hem = "E" if lon >= 0 else "W"

    lat_str = f'{lat_d}°{lat_m}\'{lat_s}"{lat_hem}'
    lon_str = f'{lon_d}°{lon_m}\'{lon_s}"{lon_hem}'
    return f"{lat_str}, {lon_str}"

In [78]:
def get_well_coordinates(qcode):
    query = f"""
    SELECT ?coord WHERE {{
      wd:{qcode} wdt:P625 ?coord .
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()
    bindings = results["results"]["bindings"]

    if not bindings:
        return None, None

    wkt = bindings[0]["coord"]["value"]  # "Point(lon lat)"
    try:
        lon_str, lat_str = wkt.replace("Point(", "").replace(")", "").split()
        lat, lon = float(lat_str), float(lon_str)
        dms = dms_string(lat, lon)
        return wkt, dms
    except:
        return None, None

def save_coordinates_to_csv(qcode, filename):
    point, dms = get_well_coordinates(qcode)
    if not point or not dms:
        print(f"No coordinates found for {qcode}")
        return

    file_exists = os.path.isfile(filename)
    with open(filename, mode="a", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["subject_q", "object-point", "object-coords"])
        writer.writerow([qcode, point, dms])
    print(f"Saved: {qcode}, {point}, {dms}")

In [79]:
# well coordinates - modelled in protege
save_coordinates_to_csv(target_qcode, "coordinates.csv")

Saved: Q126456441, Point(-7.0681791 52.4892388), 52°29'21.2597"N, 7°4'5.4448"W


In [ ]:
#mouth of the watercourse - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P403",
    csv_file=csv_file
)
obj_qs = df['object_q'].dropna().unique().tolist()
if obj_qs:
    vals = ' '.join(f"wd:{q}" for q in obj_qs)
    sparql.setQuery(f"""
    SELECT ?item ?coord WHERE {{
      VALUES ?item {{ {vals} }}
      ?item wdt:P625 ?coord .
    }}
    """)
    res = sparql.query().convert()

    point_map = {}
    dms_map = {}
    for b in res['results']['bindings']:
        q = b['item']['value'].rsplit('/', 1)[-1]
        wkt = b['coord']['value']  # e.g. Point(-7.06818 52.48924)
        point_map[q] = wkt
        match = re.match(r'Point\(([-\d.]+) ([-\d.]+)\)', wkt)
        if match:
            lon, lat = map(float, match.groups())
            dms_map[q] = dms_string(lat, lon)
else:
    point_map = {}
    dms_map = {}

# Append both formats to the dataframe
df['object-point'] = df['object_q'].map(point_map)
df['object-coords'] = df['object_q'].map(dms_map)

# Save result
df.to_csv("mouth_of_watercourse.csv", index=False)

#### IDS - Columbkilles example

In [80]:
#all properties that are instance of something that is subclass of unique id(Q6545185) - modelled in protege
# returns a bit too much weird check again.
def get_ids(target_qcode):
    query = f"""
    SELECT ?prop ?propLabel ?value WHERE {{
      wd:{target_qcode} ?p ?value.

      FILTER(STRSTARTS(STR(?p), STR(wdt:)))

      BIND(IRI(REPLACE(STR(?p), STR(wdt:), STR(wd:))) AS ?prop)

      ?prop wdt:P31 / wdt:P279* wd:Q6545185.

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()

    output = []
    for result in results["results"]["bindings"]:
        row = {
            "subject_q": target_qcode,
            "p_label": result.get("propLabel", {}).get("value", ""),
            "predicate": result.get("prop", {}).get("value", "").split("/")[-1],
            "value": result.get("value", {}).get("value", "")
        }
        output.append(row)

    return pd.DataFrame(output, columns=[
        "subject_q", "p_label", "predicate", "value"
    ])


In [81]:
df = get_ids(target_qcode)
df.to_csv("ids.csv", index=False)

#### Others

In [93]:
def extract_entity_relation_info(subject_qcode, predicate_pcode):
    # Helpers
    def extract_q(uri):
        return re.search(r'Q\d+', uri).group(0) if isinstance(uri, str) and re.search(r'Q\d+', uri) else None

    # 1. Query subject → predicate → object triples
    sparql.setQuery(f"""
    SELECT ?object WHERE {{
      wd:{subject_qcode} wdt:{predicate_pcode} ?object .
    }}
    """)
    results = sparql.query().convert()
    objects = [extract_q(r["object"]["value"]) for r in results["results"]["bindings"]]
    objects = [o for o in objects if o is not None]

    if not objects:
        return pd.DataFrame(columns=['subject_q', 'subject', 'object_q', 'object', 'instance_of_q', 'instance_of'])

    # 2. Query instance_of (P31) for all objects
    objects_str = " ".join(f"wd:{o}" for o in objects)
    sparql.setQuery(f"""
    SELECT ?obj ?inst WHERE {{
      VALUES ?obj {{ {objects_str} }}
      OPTIONAL {{ ?obj wdt:P31 ?inst. }}
    }}
    """)
    results = sparql.query().convert()
    instance_data = []
    for r in results["results"]["bindings"]:
        obj_q = extract_q(r["obj"]["value"])
        inst_q = extract_q(r["inst"]["value"]) if "inst" in r else ""
        instance_data.append((obj_q, inst_q))

    instance_df = pd.DataFrame(instance_data, columns=["object_q", "instance_of_q"])

    # 3. Fetch labels for all Q-codes
    all_qcodes = {subject_qcode} | set(objects) | set(instance_df['instance_of_q'].dropna())
    qcodes_str = " ".join(f"wd:{q}" for q in all_qcodes if q)

    sparql.setQuery(f"""
    SELECT ?item ?itemLabel WHERE {{
      VALUES ?item {{ {qcodes_str} }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """)
    results = sparql.query().convert()
    q_dict = {extract_q(r["item"]["value"]): r["itemLabel"]["value"] for r in results["results"]["bindings"]}

    # 4. Assemble final table
    links = instance_df.copy()
    links["subject_q"] = subject_qcode
    links["subject"] = q_dict.get(subject_qcode, subject_qcode)
    links["object"] = links["object_q"].apply(lambda q: q_dict.get(q, q))
    links["instance_of"] = links["instance_of_q"].apply(lambda q: q_dict.get(q, q) if pd.notnull(q) and q != "" else "")

    final_df = links[['subject_q', 'subject', 'object_q', 'object', 'instance_of_q', 'instance_of']]
    return final_df



In [95]:
#described by source - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P1343",
)
df.to_csv("described_by_source.csv", index=False)

In [ ]:
#TODO: query for info of the source itself.

In [111]:
# located in the administrative territorial entity - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P131"
)
df.to_csv("located_in_the_administrative_territorial_entity.csv", index=False)

In [98]:
#named after - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P138"
  
)
df.to_csv("named_after.csv", index=False)

In [ ]:
## query named after the wrong thing but still have a feast day
#SELECT ?holyWell ?holyWellLabel ?namedAfter ?namedAfterLabel ?feastDay
#WHERE {
  # Holy Well instances
#  ?holyWell wdt:P31 wd:Q126443332 .
  
  # Named after some entity
#  ?holyWell wdt:P138 ?namedAfter .
  
  # Feast day
#  ?holyWell wdt:P841 ?feastDay .
  
  # Exclude namedAfter entities that are instance of the forbidden list
 # FILTER NOT EXISTS {
  #  ?namedAfter wdt:P31 ?forbidden .
  #  VALUES ?forbidden { wd:Q20643955 wd:Q5 wd:Q60075825 wd:Q4818719 wd:Q4927045  }
  #}
  
  #SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
#}


In [100]:
#location - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P276"
)
df.to_csv("location.csv", index=False)

In [101]:
#diocese - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P708"
 
)
df.to_csv("diocese.csv", index=False)

In [102]:
#feast day - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P841"
)
df.to_csv("feast_day.csv", index=False)
#TODO if RRULE is ok, change Month name to month number

In [103]:
#country - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P17"

)
df.to_csv("country.csv", index=False)

In [104]:
#historic county - Lowrys Well - model with falls within. are there any time constraints?
df = extract_entity_relation_info(
    subject_qcode="Q131625906",
    predicate_pcode="P7959"
)
df.to_csv("historic_county.csv", index=False)

In [105]:
# medical condition treated - modeled in protege with P15i influenced
df = extract_entity_relation_info(
    subject_qcode="Q126472950",
    predicate_pcode="P2175"
)
df.to_csv("medical_treated.csv", index=False)


In [106]:
#medical condition - modeled in protege with P15i influenced
df = extract_entity_relation_info(
    subject_qcode="Q122302608",
    predicate_pcode="P1050"
)
df.to_csv("medical_condition.csv", index=False)

In [107]:
#significant person - example on st lachtain  - modeled in protege with p15i influenced
df = extract_entity_relation_info(
    subject_qcode="Q121840779",
    predicate_pcode="P3342"
)
df.to_csv("significant_person.csv", index=False)


In [108]:
#patron saint - example on Hermit well - modelled in protege
df = extract_entity_relation_info(
    subject_qcode="Q126478185",
    predicate_pcode="P417"
)
df.to_csv("patron_saint.csv", index=False)

In [109]:
#use state as direct properties -- ST Mogues - modeled in protege with P44
df = extract_entity_relation_info(
    subject_qcode="Q126454603",
    predicate_pcode="P5817"
)
df.to_csv("use_state.csv", index=False)

In [110]:
#conservation state as direct properties -- ST Augustine
df = extract_entity_relation_info(
    subject_qcode="44",
    predicate_pcode="P5816"
)
df.to_csv("conservation_state.csv", index=False)
